In [1]:
import pandas as pd
from eyened_orm.db import Database
from tqdm.notebook import tqdm
from pathlib import Path

## Feature normalisation and derived features

This is a sample notebook showing one way to post-process VascX outputs. While several vascx biomarkers such as tortuosity are unit-less, others such as vessel calibers or CREs measure distances. VascX, by default produces distance measurements in pixels and leaves the potential conversion of such measurements up to the user.

This notebooks normalizes these metrics by dividing by the distance between optic disc and fovea (also in pixels). Additionally, it computes artery-vein ratios from the central retinal equivalents and adds them to the biomarker set.

Consider using as a post-processing step if this normalisation strategy works in your analysis.

Important: may need to be adapted for feature sets other than `full_v3`

In [ ]:
FEAT_PATH = Path("../samples/fundus/biomarkers.csv")
OUTPUT_PATH = Path("../samples/fundus/biomarkers_normalized.csv")
df = pd.read_csv(FEAT_PATH, index_col=0)

In [ ]:
# Normalize all CRE and diameter features by optic disc diameter
cre_features = [col for col in df.columns if '_cre_' in col]
diam_features = [col for col in df.columns if '_diam_' in col]
vd_features = [col for col in df.columns if 'vd_' in col]
rest_cols = sorted(list(set(df.columns) - set(cre_features) - set(diam_features)))

for col in cre_features + diam_features:
    df[col] = df[col] / df['disc_fovea_distance_retina']

In [18]:
print('Caliber biomarkers (OD-fovea-normalised):')
for col in diam_features:
    print(f'- {col}')
print('Central retinal equivalents: (OD-fovea-normalised)')
for col in cre_features:
    print(f'- {col}')
print('Remaining features (no additional normalisation): ')
for col in rest_cols:
    print(f'- {col}')

Caliber biomarkers (OD-fovea-normalised):
- md_diam_arteries
- md_diam_veins
- md_diam_hf_superior_arteries
- md_diam_hf_superior_veins
- md_diam_hf_inferior_arteries
- md_diam_hf_inferior_veins
- lw_diam_arteries
- lw_diam_veins
- lw_diam_hf_superior_arteries
- lw_diam_hf_superior_veins
- lw_diam_hf_inferior_arteries
- lw_diam_hf_inferior_veins
- lw_diam_disc_full_arteries
- lw_diam_disc_full_veins
Central retinal equivalents: (OD-fovea-normalised)
- temporal_cre_arteries
- temporal_cre_veins
- temporal_cre_hf_superior_arteries
- temporal_cre_hf_superior_veins
- temporal_cre_hf_inferior_arteries
- temporal_cre_hf_inferior_veins
- nasal_cre_arteries
- nasal_cre_veins
- full_cre_arteries
- full_cre_veins
Remaining features (no additional normalisation): 
- disc_fovea_distance_retina
- lw_tort_curv_arteries
- lw_tort_curv_disc_full_arteries
- lw_tort_curv_disc_full_veins
- lw_tort_curv_etdrs_full_arteries
- lw_tort_curv_etdrs_full_veins
- lw_tort_curv_veins
- lw_tort_dist_arteries
- lw_t

In [12]:
# add the artery-vein ratios
for col in ['temporal_cre', 'temporal_cre_hf_superior', 'temporal_cre_hf_inferior', 'nasal_cre', 'full_cre']:
    df[f'avr_{col}'] = df[f'{col}_arteries'] / df[f'{col}_veins']

## Re-scale features (optional)

CREs and diameters are very small after division. Vascular densities can also be converted to percentages.

In [ ]:
for col in cre_features + diam_features + vd_features:
    df[col] = df[col] * 100

In [ ]:
df.to_csv(OUTPUT_PATH)